# 修复说明（2026-09-29）

这份是 `4. delay_modeling.ipynb` 的修复版，改了以下几点：

1. **Mean/MICE 结果重复的 bug**：原来 `record_downstream_result` 调 Mean 那次时，用的其实是上一个 cell（MICE）留下的 `y_pred`/`y_prob`，导致两行数值完全一样。现在改成统一的 `run_model_and_record()` 函数，训练/预测/记录在同一次调用里原子完成，不依赖任何跨 cell 残留变量。
2. **`rf` 未定义会直接报错**：原 Median 那格代码写的是 `xgb_model = XGBClassifier(...)` 却调用 `rf.fit()`，而 `rf` 在那之前根本没定义过——只有靠手动乱序执行（先跑了 KNN 那格）才凑巧不报错。Restart & Run All 会直接 `NameError`。现在统一用同一个模型，不存在这个问题。
3. **模型不统一**：原来 Mean/XGBoost，Median/未定义、KNN 和 MICE/RandomForest(balanced)。四种方法现在统一用 `RandomForestClassifier(class_weight="balanced")` —— 选它是因为 XGBoost（没设 `scale_pos_weight`）在两份 notebook 里都只会预测多数类（Precision=Recall=0），RF balanced 至少能学到一点少数类。如果你想同时对比 XGBoost，可以把 `MODEL` 那格换掉重跑一次。
4. **对齐方式**：原来是 `mcar_10.index = model_df.index` 直接把 index 硬套上去（假设 parquet 读回来的行顺序没变）。现在改成用 `order_id` 显式过滤 train/test，不依赖行顺序假设。
5. **新增两个 baseline**（来自你的 `model_test.ipynb`，用同一套统一函数重跑，方便和其他结果放进同一张表比较）：
   - **Full data**：完全没有缺失值注入，直接用干净数据训练
   - **Drop column**：训练时直接不用 `total_freight_value` 这一列

跑完之后麻烦 Restart & Run All 确认一遍，最后 `downstream_results_df` 应该是 36 + 2 = 38 行。


# Library

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


# Load Clean Dataset

In [2]:
df = pd.read_parquet(
    "/Users/sivhoung/Desktop/Year 4 intern project/Project code /dataset/processed dataset/final_df_(distance).parquet"
)

final_features = [
    "total_price",
    "total_freight_value",
    "total_product_weight_g",
    "average_product_length_cm",
    "average_product_height_cm",
    "item_count",
    "max_payment_installments",
    "average_distance_km"
]

target_col = "total_freight_value"

experiment_columns = ["order_id", *final_features, "is_delayed"]

model_df = df[experiment_columns].dropna().copy()

print(model_df.shape)
print(model_df.isna().sum().sum())
print("order_id unique:", model_df["order_id"].is_unique)


(95960, 10)
0
order_id unique: True


In [3]:
train_df, test_df = train_test_split(
    model_df,
    test_size=0.2,
    random_state=42,
    stratify=model_df["is_delayed"]
)

train_order_ids = set(train_df["order_id"])
test_order_ids = set(test_df["order_id"])

print("Train:", train_df.shape)
print("Test:", test_df.shape)


Train: (76768, 10)
Test: (19192, 10)


# Load Missing Dataset

In [4]:
data_folder = Path("/Users/sivhoung/Desktop/Year 4 intern project/Project code /dataset/exp_data")

injection_data = {}

for file in data_folder.glob("*.parquet"):
    injection_data[file.stem] = pd.read_parquet(file)

print(sorted(injection_data.keys()))


['mar_10', 'mar_30', 'mar_50', 'mcar_10', 'mcar_30', 'mcar_50', 'mnar_10', 'mnar_30', 'mnar_50']


## 用 order_id 做 train/test 划分（替代原来的 index 硬套）

原来是 `mcar_10.index = model_df.index`，假设 parquet 读回来的行顺序跟 model_df 完全一致。
这里改成显式按 `order_id` 过滤，不依赖行顺序假设，更稳妥。


In [5]:
def split_by_order_id(injected_df):
    inj_train = injected_df[injected_df["order_id"].isin(train_order_ids)].copy()
    inj_test = injected_df[injected_df["order_id"].isin(test_order_ids)].copy()
    return inj_train, inj_test


# 统一的填补函数（train-only 统计量，test 只 transform，不泄漏）

In [6]:
def impute_mean(train_df, test_df, target):
    train_out, test_out = train_df.copy(), test_df.copy()
    value = train_out[target].mean()
    train_out[target] = train_out[target].fillna(value)
    test_out[target] = test_out[target].fillna(value)
    return train_out, test_out


def impute_median(train_df, test_df, target):
    train_out, test_out = train_df.copy(), test_df.copy()
    value = train_out[target].median()
    train_out[target] = train_out[target].fillna(value)
    test_out[target] = test_out[target].fillna(value)
    return train_out, test_out


def impute_knn(train_df, test_df, features, n_neighbors=5):
    train_out, test_out = train_df.copy(), test_df.copy()

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(train_out[features])
    X_test_scaled = scaler.transform(test_out[features])

    imputer = KNNImputer(n_neighbors=n_neighbors, weights="uniform")
    X_train_imp_scaled = imputer.fit_transform(X_train_scaled)
    X_test_imp_scaled = imputer.transform(X_test_scaled)

    train_out[features] = scaler.inverse_transform(X_train_imp_scaled)
    test_out[features] = scaler.inverse_transform(X_test_imp_scaled)
    return train_out, test_out


def impute_mice(train_df, test_df, features, n_iter=10, random_seed=42):
    train_out, test_out = train_df.copy(), test_df.copy()

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(train_out[features])
    X_test_scaled = scaler.transform(test_out[features])

    imputer = IterativeImputer(max_iter=n_iter, random_state=random_seed)
    X_train_imp_scaled = imputer.fit_transform(X_train_scaled)
    X_test_imp_scaled = imputer.transform(X_test_scaled)

    train_out[features] = scaler.inverse_transform(X_train_imp_scaled)
    test_out[features] = scaler.inverse_transform(X_test_imp_scaled)
    return train_out, test_out


# 统一的模型 + 记录函数（原子操作，不依赖跨 cell 残留变量）

In [7]:
def run_model_and_record(results, train_df, test_df, mechanism, missing_rate, method,
                          features=final_features):
    X_train = train_df[features]
    y_train = train_df["is_delayed"]
    X_test = test_df[features]
    y_test = test_df["is_delayed"]

    model = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "mechanism": mechanism,
        "missing_rate": missing_rate,
        "method": method,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob)
    })


# 主循环：9 个注入数据集 × 4 种填补方法 = 36 行

In [8]:
downstream_results = []

for name, injected_df in injection_data.items():
    mechanism, rate_str = name.split("_")
    mechanism = mechanism.upper()
    rate = int(rate_str) / 100

    inj_train, inj_test = split_by_order_id(injected_df)

    # Mean
    m_train, m_test = impute_mean(inj_train, inj_test, target_col)
    run_model_and_record(downstream_results, m_train, m_test, mechanism, rate, "Mean")

    # Median
    m_train, m_test = impute_median(inj_train, inj_test, target_col)
    run_model_and_record(downstream_results, m_train, m_test, mechanism, rate, "Median")

    # KNN
    m_train, m_test = impute_knn(inj_train, inj_test, final_features)
    run_model_and_record(downstream_results, m_train, m_test, mechanism, rate, "KNN")

    # MICE
    m_train, m_test = impute_mice(inj_train, inj_test, final_features)
    run_model_and_record(downstream_results, m_train, m_test, mechanism, rate, "MICE")

downstream_results_df = pd.DataFrame(downstream_results)
downstream_results_df = downstream_results_df.drop_duplicates().sort_values(
    ["mechanism", "missing_rate", "method"]
).reset_index(drop=True)

print("Rows:", len(downstream_results_df))
downstream_results_df


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: in

Rows: 36


,mechanism,missing_rate,method,accuracy,precision,recall,f1,roc_auc
0,MAR,0.1,KNN,0.915642,0.203704,0.014148,0.026458,0.590650
1,MAR,0.1,MICE,0.915694,0.216216,0.015434,0.028812,0.587130
2,MAR,0.1,Mean,0.915486,0.192661,0.013505,0.025240,0.585981
3,MAR,0.1,Median,0.915955,0.191489,0.011576,0.021831,0.579000
4,MAR,0.3,KNN,0.916111,0.204301,0.012219,0.023058,0.580634
5,MAR,0.3,MICE,0.916163,0.193182,0.010932,0.020694,0.576216
6,MAR,0.3,Mean,0.915955,0.191489,0.011576,0.021831,0.575290
7,MAR,0.3,Median,0.916007,0.186813,0.010932,0.020656,0.578807
8,MAR,0.5,KNN,0.916111,0.222222,0.014148,0.026602,0.569781
9,MAR,0.5,MICE,0.916580,0.244444,0.014148,0.026748,0.569139


# Baseline 1：Full data（完全没有缺失值注入）

用同一套 `run_model_and_record`，直接在 `train_df`/`test_df`（干净数据）上跑，
方便和上面 36 行放进同一张表比较——用来判断"填补方法的选择到底有没有实质影响"。


In [9]:
run_model_and_record(
    downstream_results,
    train_df, test_df,
    mechanism="Baseline", missing_rate=0.0, method="Full_data"
)


# Baseline 2：Drop column（训练时直接不用 total_freight_value）

如果 drop 掉这一列跟"填补后再用"效果差不多，说明这个特征/填补方法对模型本身贡献有限。


In [10]:
features_no_target = [f for f in final_features if f != target_col]

run_model_and_record(
    downstream_results,
    train_df, test_df,
    mechanism="Baseline", missing_rate=0.0, method="Drop_column",
    features=features_no_target
)

downstream_results_df = pd.DataFrame(downstream_results)
downstream_results_df = downstream_results_df.drop_duplicates().sort_values(
    ["mechanism", "missing_rate", "method"]
).reset_index(drop=True)

print("Total rows:", len(downstream_results_df))
downstream_results_df


Total rows: 38


,mechanism,missing_rate,method,accuracy,precision,recall,f1,roc_auc
0,Baseline,0.0,Drop_column,0.913401,0.163522,0.016720,0.030338,0.573224
1,Baseline,0.0,Full_data,0.914808,0.161017,0.012219,0.022714,0.595768
2,MAR,0.1,KNN,0.915642,0.203704,0.014148,0.026458,0.590650
3,MAR,0.1,MICE,0.915694,0.216216,0.015434,0.028812,0.587130
4,MAR,0.1,Mean,0.915486,0.192661,0.013505,0.025240,0.585981
5,MAR,0.1,Median,0.915955,0.191489,0.011576,0.021831,0.579000
6,MAR,0.3,KNN,0.916111,0.204301,0.012219,0.023058,0.580634
7,MAR,0.3,MICE,0.916163,0.193182,0.010932,0.020694,0.576216
8,MAR,0.3,Mean,0.915955,0.191489,0.011576,0.021831,0.575290
9,MAR,0.3,Median,0.916007,0.186813,0.010932,0.020656,0.578807


# 保存结果

In [11]:
downstream_results_df.to_csv(
    "/Users/sivhoung/Desktop/Year 4 intern project/Project code /results/downstream_results2.csv",
    index=False
)
